<a href="https://colab.research.google.com/github/AshrfCode/Anan-Tirgulem/blob/main/tutorial1/tirgul1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
from google.colab import drive

# 1. חיבור לדרייב
drive.mount('/content/drive')

# 2. הגדרת נתיב הקובץ
FOLDER_NAME = '/content/drive/MyDrive/Anan/tergol1'
FILE_PATH = f'{FOLDER_NAME}/students.txt'

# נוודא שהתיקייה קיימת
os.makedirs(FOLDER_NAME, exist_ok=True)

# פונקציה לטעינת הסטודנטים מהקובץ
def load_students():
    students_data = {}
    if not os.path.exists(FILE_PATH):
        print(f"⚠️ לא נמצא קובץ בנתיב {FILE_PATH}. נא לוודא שהקובץ קיים.")
        return students_data

    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(',')]
            if len(parts) >= 5:
                full_name = f"{parts[0]} {parts[1]}"
                # אם יש לסטודנט מספר שורות, השורה האחרונה בקובץ תדרוס את הקודמת במילון
                # וכך תמיד נציג את המידע המעודכן ביותר
                students_data[full_name] = {
                    'first_name': parts[0],
                    'last_name': parts[1],
                    'email': parts[2],
                    'courses': parts[3],
                    'link_or_show': parts[4] # זהו הקישור המעניין / תוכנית אהובה
                }
    return students_data

students = load_students()

# --- עיצוב ויצירת רכיבי הטופס ---
style = {'description_width': '130px'}
layout = widgets.Layout(width='450px', margin='5px 0')

title = widgets.HTML("<h2 style='color: #4285F4; text-align: right; direction: rtl;'>🎓 טופס ניהול סטודנטים</h2>")

options_list = ['...בחר סטודנט'] + list(students.keys()) if students else ['...בחר סטודנט']
dropdown = widgets.Dropdown(options=options_list, description='👤 בחר סטודנט:', style=style, layout=layout)

# תיבות טקסט לפרטים אישיים (מושבתות לעריכה - מוצגות לקריאה בלבד)
txt_first = widgets.Text(description='שם פרטי:', style=style, layout=layout, disabled=True)
txt_last = widgets.Text(description='שם משפחה:', style=style, layout=layout, disabled=True)
txt_email = widgets.Text(description='📧 אימייל:', style=style, layout=layout, disabled=True)
txt_courses = widgets.Text(description='📚 קורסים:', style=style, layout=layout, disabled=True)

# תיבת הטקסט לעדכון תוכנית אהובה / קישור מעניין (היחידה שפתוחה לעריכה!)
txt_link_show = widgets.Text(description='📺 תוכנית/קישור:', style=style, layout=layout, placeholder='הזן תוכנית אהובה או קישור...')

btn_update = widgets.Button(description='💾 עדכן נתונים', button_style='success', icon='check', layout=widgets.Layout(width='200px', margin='15px 120px 0 0'))
output_msg = widgets.Output()

# --- פונקציות פעולה ---
def on_student_select(change):
    if change['new'] != '...בחר סטודנט' and change['new'] in students:
        data = students[change['new']]
        txt_first.value = data['first_name']
        txt_last.value = data['last_name']
        txt_email.value = data['email']
        txt_courses.value = data['courses']
        # מציג את התוכנית/קישור הנוכחי ששמור בקובץ
        txt_link_show.value = data['link_or_show']
    else:
        for txt in [txt_first, txt_last, txt_email, txt_courses, txt_link_show]:
            txt.value = ''

dropdown.observe(on_student_select, names='value')

def on_update_click(b):
    with output_msg:
        clear_output()
        if dropdown.value == '...בחר סטודנט':
            print("❌ אנא בחר סטודנט תחילה.")
            return

        # מרכיב את השורה החדשה עם הנתונים הקיימים + הקישור/תוכנית החדשים שהוזנו בתיבה
        new_line = f"{txt_first.value}, {txt_last.value}, {txt_email.value}, {txt_courses.value}, {txt_link_show.value}\n"

        # מוסיף את השורה לקובץ (במצב 'a' - Append) מבלי למחוק את הישנה
        with open(FILE_PATH, 'a', encoding='utf-8') as f:
            f.write(new_line)

        # מעדכן את הנתונים בזיכרון כדי שהשינוי ישתקף מיד בטופס
        students[dropdown.value]['link_or_show'] = txt_link_show.value
        print(f"✅ מעולה! השורה החדשה עם התוכנית/הקישור המעודכן התווספה לקובץ בדרייב.")

btn_update.on_click(on_update_click)

# --- סידור התצוגה ---
form_ui = widgets.VBox([
    title,
    dropdown,
    txt_first,
    txt_last,
    txt_email,
    txt_courses,
    txt_link_show, # שדה העדכון
    btn_update,
    output_msg
])

# הצגת הטופס
display(form_ui)